In [0]:
import requests
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import current_timestamp, col, round


# 1. Automatic Ingestion from API
print("Step 1: Ingesting data from API...")
url = "https://api.coingecko.com/api/v3/coins/markets"
params = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 50,
    "page": 1,
    "sparkline": "false"
}


response = requests.get(url, params=params, timeout=10)
data = response.json()


pdf = pd.DataFrame(data)
df_api = spark.createDataFrame(pdf)


# 2. Medallion Architecture: Bronze Layer
print("Step 2: Processing Bronze Layer (Delta Lake)...")


df_bronze = df_api.withColumn("ingested_at", current_timestamp())


df_bronze.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("crypto_bronze")

print("Bronze Table updated successfully!")


# 3. Medallion Architecture: Silver Layer
print("Step 3: Processing Silver Layer (Cleaned Data)...")

df_silver = spark.table("crypto_bronze") \
    .select(
        col("id").alias("coin_id"),
        col("symbol"),
        col("name"),
        round(col("current_price").cast("double"), 2).alias("price_usd"),
        col("market_cap").cast("long"),
        col("total_volume").cast("long"),
        col("ingested_at")
    ) \
    .filter(col("price_usd").isNotNull())


df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("crypto_silver")

print("Pipeline execution completed successfully!")


# 4. Query & Verify Silver Table Results
display(spark.sql("SELECT * FROM crypto_silver ORDER BY market_cap DESC LIMIT 10"))

Step 1: Ingesting data from API...
Step 2: Processing Bronze Layer (Delta Lake)...
Bronze Table updated successfully!
Step 3: Processing Silver Layer (Cleaned Data)...
Pipeline execution completed successfully!


coin_id,symbol,name,price_usd,market_cap,total_volume,ingested_at
bitcoin,btc,Bitcoin,79234.0,1591057428275,24406769490,2026-09-07T19:33:39.685Z
bitcoin,btc,Bitcoin,79182.0,1590070237201,24396590569,2026-09-07T18:25:04.382Z
bitcoin,btc,Bitcoin,79167.0,1589786965943,24371356009,2026-09-07T17:58:33.561Z
bitcoin,btc,Bitcoin,78429.0,1575257213611,26000586044,2026-09-08T07:25:16.997Z
ethereum,eth,Ethereum,2495.21,304473164899,11979787307,2026-09-07T17:58:33.561Z
ethereum,eth,Ethereum,2494.64,304401230778,11988870755,2026-09-07T18:25:04.382Z
ethereum,eth,Ethereum,2492.2,304119629163,11881912286,2026-09-07T19:33:39.685Z
ethereum,eth,Ethereum,2475.02,302070260296,10910530900,2026-09-08T07:25:16.997Z
tether,usdt,Tether,1.0,183400160986,50274948688,2026-09-08T07:25:16.997Z
tether,usdt,Tether,1.0,183393978123,49455518050,2026-09-07T18:25:04.382Z
